# Self-Supervised 2D U-Net Training

This notebook runs the two-stage pipeline for pediatric brain tumor segmentation:

- self-supervised masked reconstruction pretraining
- supervised fine-tuning on the labeled subset
- validation and test-time prediction export


## 1. Setup Modules


In [ ]:
# !pip install -q torch torchvision torchaudio
# !pip install -q numpy tqdm

## 2. Mount Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## 3. Configure project paths and hyperparameters


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/School/ECE324/code')

DATA_ROOT = PROJECT_ROOT / 'data'
MANIFEST_CSV = PROJECT_ROOT / 'ssl_split_manifest_20.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'ssl_unet2d_colab'

PRETEXT_TASK = 'cross_modal'  # 'cross_modal' or 'random_channel_dropout'
PRETRAIN_EPOCHS = 20
FINETUNE_EPOCHS = 30
BATCH_SIZE = 4
NUM_WORKERS = 2
MASK_RATIO = 0.5
PATCH_SIZE = 16
SEED = 42

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATA_ROOT =', DATA_ROOT)
print('MANIFEST_CSV =', MANIFEST_CSV)
print('OUTPUT_DIR =', OUTPUT_DIR)

## 4. Import the training pipeline

make sure the current directory include files: `unet2d.py`, `load_dataset.py`, `pretraining.py`, `fine_tunning.py`, `utils.py`, and `start_training.py`.

In [ ]:
import os
import sys

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from run_training import build_arg_parser, run_pipeline

## 5. Build arguments

This cell mirrors the command-line options from `start_training.py`.

In [ ]:
parser = build_arg_parser()
args = parser.parse_args(args=[])

args.data_root = str(DATA_ROOT)
args.manifest_csv = str(MANIFEST_CSV)
args.output_dir = str(OUTPUT_DIR)
args.pretext_task = PRETEXT_TASK
args.pretrain_epochs = PRETRAIN_EPOCHS
args.finetune_epochs = FINETUNE_EPOCHS
args.batch_size = BATCH_SIZE
args.num_workers = NUM_WORKERS
args.mask_ratio = MASK_RATIO
args.patch_size = PATCH_SIZE
args.seed = SEED

args

## 6. Run training

This will pretrain, fine-tune, evaluate on the test set, and save predicted masks.

In [ ]:
run_pipeline(args)

## 7. Inspect outputs

The best checkpoints and test predictions are saved under `OUTPUT_DIR`.

In [ ]:
from pathlib import Path

for path in sorted(Path(OUTPUT_DIR).glob('*')):
    print(path)